# Step 3.7: combinators and features inside a strategy

This notebook is the check of step 3.7. Features now combine into new features without code (`ff.features.combine`), and a strategy reads features from your library by name and version, which every run names in its record. Here you build `rsi_stretch`, RSI(14) less its mean over ten bars, with the combinators, check it with the leak detector, save it, and backtest a strategy of it and of RSI(14), both read back from your library. Running the strategy again opens the saved run without loading its bars or evaluating its signals.

Pick the `.venv` kernel and run all cells. It reads the store in `FEFAT_DATA_DIR`, and keeps the features in your library and the runs under `FEFAT_HOME`.

## A feature built from others

The product is MES when your store holds its 1-minute bars, else the product with the most contracts, and the window the two weeks to its last trade date. `combine.define` writes the feature's code from the combination: it reads RSI(14) with its parameters, declares a warm-up of nine bars for its mean, and gives a ratio.

Expected: the product and the window, the feature's code, and its description, a `subtract` of a `feature` and a `rolling_mean`.

In [ ]:
import time
from datetime import timedelta

import polars as pl

import fefat as ff
from fefat.features import combine

listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, LAST = chosen["root"], chosen["last_trade_date"]
START = LAST - timedelta(days=14)
rsi = ff.indicators.rsi
rsi_stretch = combine.define("rsi_stretch", rsi(14) - combine.rolling_mean(rsi(14), 10))
description = rsi_stretch.combination.model_dump(mode="json")
parts = f"{description['left']['kind']} and a {description['right']['kind']}"
print(f"{ROOT} 5-minute bars from {START} to {LAST}")
print(rsi_stretch.code)
print(f"Its description: a {description['kind']} of a {parts}")

## The leak detector

A combination is a feature as any other, so `ff.features.validate` checks it as it checks one you write: its values before each cut of the data are those of the whole range, the window's start can move, and the lint finds nothing in its code. The report names RSI(14), which the combination read without being given.

Expected: `PASSED`, and the feature it read with its parameters.

In [ ]:
report = ff.features.validate(rsi_stretch(), ROOT, "5m", START, LAST)
read = ", ".join(f"{each.name} {each.parameters}" for each in report.reads)
print(f"The leak detector on {ROOT}'s 5-minute bars: {'PASSED' if report.passed else 'FAILED'}")
print(f"It read: {read}")

## Your library

`ff.features.save` keeps the feature's version in your library, and first the RSI it reads; `ff.features.library()` lists every saved version.

Expected: `rsi` and `rsi_stretch` with their versions, `rsi_stretch` a combination.

In [ ]:
saved = ff.features.save(rsi_stretch)
library = ff.features.library()
RSI_VERSION = library.filter(pl.col("hash") == rsi.hash).item(0, "version")
mine = library.filter(pl.col("name").is_in(["rsi", "rsi_stretch"]))
for name, version, combined in mine.select("name", "version", "combined").iter_rows():
    print(f"  {name} version {version}{', a combination' if combined else ''}")

## A strategy of two library features

The strategy reads both features from your library by name and version, `ff.features.ref`, and goes long where the RSI stretches above its mean while above 50, short where it stretches below while under 50. Its run saves the features it reads and names each one's version in its record, `run.json`.

Expected: the run's trades, each feature's slot, name, version, hash, and parameters as the run names them, those your library keeps, and `PASSED`.

In [ ]:
strategy = ff.SignalStrategy(
    instrument=ROOT,
    timeframe="5m",
    features={
        "stretch": ff.features.ref("rsi_stretch", version=saved.version),
        "rsi": ff.features.ref("rsi", version=RSI_VERSION, n=14),
    },
    long_entry=(pl.col("stretch") > 5) & (pl.col("rsi") > 50),
    short_entry=(pl.col("stretch") < -5) & (pl.col("rsi") < 50),
    execution=ff.Execution(
        entry=ff.Market(), stop=ff.Stop(ticks=16), targets=[ff.Target(ticks=32)], sizing=ff.Fixed(1)
    ),
)
started = time.perf_counter()
run = ff.backtest.run(strategy, START, LAST)
FIRST = time.perf_counter() - started
pins = run.record.strategy.feature_versions
window = run.record.window
print(f"{run.record.summary.trades} trades from {window.start} to {window.end}")
for pin in pins:
    print(f"  {pin.slot}: {pin.name} version {pin.version}, {pin.hash[:12]}, {pin.parameters}")
kept = set(library.select("name", "version", "hash").iter_rows())
named = {(pin.name, pin.version, pin.hash) for pin in pins}
print(f"The run names the versions your library keeps: {'PASSED' if named <= kept else 'FAILED'}")

## The same run again

A second run of the same strategy on the same data finds its request beside the saved runs and opens the saved run without loading its bars or evaluating its signals, where before it evaluated them again before it found the saved run: 1.5 s against the first's 2.2 s on 23 sessions of MES in step 2.10.

Expected: both runs' times, the second much the shorter, and the same run: `PASSED`.

In [ ]:
started = time.perf_counter()
again = ff.backtest.run(strategy, START, LAST)
SECOND = time.perf_counter() - started
print(f"First run {FIRST:.2f} s, second {SECOND:.2f} s")
print(f"The second run opened the first: {'PASSED' if again.hash == run.hash else 'FAILED'}")